# 🏗️ Treinamento de Modelos de Regressão - London Houses

Neste notebook, vamos treinar e comparar diferentes modelos de regressão para prever o valor mediano das casas.

## 1. Importações e Configurações

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib

# Adicionar o diretório src ao path para importar o pipeline de pré-processamento
sys.path.append(os.path.abspath("../src"))
from data.data_processor import build_preprocessing_pipeline

# Configurações de visualização
%matplotlib inline
sns.set_theme(style="whitegrid")
os.makedirs('../models/trained_models', exist_ok=True)

## 2. Carregamento e Divisão dos Dados

In [ ]:
# Carregar os dados
df = pd.read_csv('../data/housing.csv')

# Separar features e target
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"].copy()

# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho do treino: {X_train.shape[0]}")
print(f"Tamanho do teste: {X_test.shape[0]}")

## 3. Pré-processamento

Vamos usar o pipeline definido em `src/data/data_processor.py`.

In [ ]:
preprocessor = build_preprocessing_pipeline()
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Formato dos dados processados:", X_train_prepared.shape)

## 4. Treinamento e Avaliação de Modelos

Vamos criar uma função para treinar e avaliar cada modelo.

In [ ]:
def evaluate_model(model, name, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    
    print(f"--- {name} ---")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R2 Score: {r2:.4f}")
    return rmse, r2

results = {}

# 1. Linear Regression
results['Linear Regression'] = evaluate_model(LinearRegression(), "Linear Regression", X_train_prepared, y_train, X_test_prepared, y_test)

# 2. Decision Tree
results['Decision Tree'] = evaluate_model(DecisionTreeRegressor(random_state=42), "Decision Tree", X_train_prepared, y_train, X_test_prepared, y_test)

# 3. Random Forest
results['Random Forest'] = evaluate_model(RandomForestRegressor(n_estimators=100, random_state=42), "Random Forest", X_train_prepared, y_train, X_test_prepared, y_test)

# 4. XGBoost
results['XGBoost'] = evaluate_model(XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42), "XGBoost", X_train_prepared, y_train, X_test_prepared, y_test)

## 5. Comparação de Performance

In [ ]:
res_df = pd.DataFrame(results, index=['RMSE', 'R2 Score']).T
res_df = res_df.sort_values(by='RMSE')

plt.figure(figsize=(10, 5))
sns.barplot(x=res_df.index, y=res_df['RMSE'])
plt.title('Comparação de RMSE entre Modelos')
plt.ylabel('RMSE')
plt.show()

res_df

## 6. Salvando o Melhor Modelo

O XGBoost ou Random Forest provavelmente será o melhor.

In [ ]:
# Supondo que o XGBoost seja o melhor
best_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
best_model.fit(X_train_prepared, y_train)

joblib.dump(best_model, '../models/trained_models/xgboost_model.pkl')
joblib.dump(preprocessor, '../models/trained_models/preprocessor.pkl')

print("Melhor modelo e preprocessor salvos em ../models/trained_models/")